# Machine Learning for Ramping Risk Prediction

**Purpose:** Build ML models to predict high-risk ramping events 1-3 hours ahead

**Business Value:** 
- Early warning system for grid operators
- Optimize reserve scheduling
- Reduce emergency dispatch costs

**Approach:**
1. Feature engineering from weather + load patterns
2. Binary classification: Will next hour have extreme ramp? (>95th percentile)
3. Regression: Predict exact ramp magnitude
4. Model comparison: Random Forest, Gradient Boosting, Neural Network
5. Feature importance analysis

In [ ]:
# Setup
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Paths
ROOT = Path('C:/Renewable-PowerGrid-Risk')
PROCESSED_DIR = ROOT / 'data' / 'processed'
FIGURES_DIR = ROOT / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load data
df = pd.read_csv(PROCESSED_DIR / 'hourly_load_renewable_merged.csv', parse_dates=['datetime'])

# Compute ramps if not present
if 'ramp_1h' not in df.columns:
    df['ramp_1h'] = df['NET_LOAD'].diff()
    df['ramp_3h'] = df['NET_LOAD'].diff(3)

print(f"Loaded {len(df):,} hours from {df['datetime'].min()} to {df['datetime'].max()}")
print(f"Data shape: {df.shape}")

## 1. Feature Engineering

In [ ]:
def create_features(df):
    """
    Create features for ML prediction of ramping events.
    
    Features include:
    - Temporal: hour, day of week, month, season
    - Lagged values: load, wind, solar at t-1, t-2, t-3
    - Recent trends: 3-hour, 6-hour changes
    - Ratios: renewable penetration, net load level
    - Volatility: rolling std of wind/solar
    """
    df_features = df.copy()
    
    # Temporal features
    df_features['hour'] = df_features['datetime'].dt.hour
    df_features['day_of_week'] = df_features['datetime'].dt.dayofweek
    df_features['month'] = df_features['datetime'].dt.month
    df_features['is_weekend'] = (df_features['day_of_week'] >= 5).astype(int)
    
    # Cyclical encoding for hour (captures 23→0 continuity)
    df_features['hour_sin'] = np.sin(2 * np.pi * df_features['hour'] / 24)
    df_features['hour_cos'] = np.cos(2 * np.pi * df_features['hour'] / 24)
    
    # Season
    df_features['season'] = df_features['month'] % 12 // 3  # 0=winter, 1=spring, 2=summer, 3=fall
    
    # Lagged features (t-1, t-2, t-3 hours ago)
    for lag in [1, 2, 3]:
        df_features[f'load_lag{lag}'] = df_features['ERCOT.LOAD'].shift(lag)
        df_features[f'wind_lag{lag}'] = df_features['ERCOT.WIND.GEN'].shift(lag)
        df_features[f'solar_lag{lag}'] = df_features['ERCOT.PVGR.GEN'].shift(lag)
        df_features[f'netload_lag{lag}'] = df_features['NET_LOAD'].shift(lag)
        df_features[f'ramp_lag{lag}'] = df_features['ramp_1h'].shift(lag)
    
    # Trend features (recent changes)
    df_features['wind_change_3h'] = df_features['ERCOT.WIND.GEN'].diff(3)
    df_features['solar_change_3h'] = df_features['ERCOT.PVGR.GEN'].diff(3)
    df_features['load_change_3h'] = df_features['ERCOT.LOAD'].diff(3)
    
    df_features['wind_change_6h'] = df_features['ERCOT.WIND.GEN'].diff(6)
    df_features['solar_change_6h'] = df_features['ERCOT.PVGR.GEN'].diff(6)
    
    # Volatility features (rolling standard deviation)
    df_features['wind_volatility_6h'] = df_features['ERCOT.WIND.GEN'].rolling(6).std()
    df_features['solar_volatility_6h'] = df_features['ERCOT.PVGR.GEN'].rolling(6).std()
    df_features['ramp_volatility_6h'] = df_features['ramp_1h'].rolling(6).std()
    
    # Ratio features
    df_features['renewable_penetration'] = (
        (df_features['ERCOT.WIND.GEN'] + df_features['ERCOT.PVGR.GEN']) / 
        df_features['ERCOT.LOAD']
    )
    df_features['wind_fraction'] = (
        df_features['ERCOT.WIND.GEN'] / 
        (df_features['ERCOT.WIND.GEN'] + df_features['ERCOT.PVGR.GEN'] + 1)  # +1 to avoid div by zero
    )
    df_features['netload_normalized'] = df_features['NET_LOAD'] / df_features['ERCOT.LOAD']
    
    # Critical period flags
    df_features['is_sunset'] = ((df_features['hour'] >= 17) & (df_features['hour'] <= 20)).astype(int)
    df_features['is_sunrise'] = ((df_features['hour'] >= 6) & (df_features['hour'] <= 9)).astype(int)
    df_features['is_summer'] = df_features['month'].isin([6, 7, 8]).astype(int)
    
    # Recent ramp magnitude (absolute)
    df_features['abs_ramp_lag1'] = df_features['ramp_1h'].shift(1).abs()
    df_features['abs_ramp_lag2'] = df_features['ramp_1h'].shift(2).abs()
    
    return df_features

df_features = create_features(df)
print(f"\nCreated features. Shape: {df_features.shape}")
print(f"\nFeature columns ({len([c for c in df_features.columns if c not in df.columns])} new):")
new_features = [c for c in df_features.columns if c not in df.columns]
for i, feat in enumerate(new_features, 1):
    print(f"  {i:2d}. {feat}")

## 2. Target Variable Definition

**Classification Task:** Predict if next hour will have extreme ramp (>95th percentile)

In [ ]:
# Define extreme ramp threshold (95th percentile)
ramp_threshold = df_features['ramp_1h'].abs().quantile(0.95)
print(f"Extreme ramp threshold (P95): {ramp_threshold:,.0f} MW/hr")

# Classification target: will NEXT hour have extreme ramp?
df_features['target_extreme_ramp'] = (df_features['ramp_1h'].abs() > ramp_threshold).astype(int)

# Shift target forward (predict next hour, not current hour)
df_features['target_extreme_ramp_next'] = df_features['target_extreme_ramp'].shift(-1)

# Regression target: actual ramp magnitude next hour
df_features['target_ramp_magnitude'] = df_features['ramp_1h'].abs().shift(-1)

# Class distribution
class_counts = df_features['target_extreme_ramp_next'].value_counts()
print(f"\nClass distribution:")
print(f"  Normal ramps (0): {class_counts.get(0, 0):,} hours ({class_counts.get(0, 0)/len(df_features)*100:.1f}%)")
print(f"  Extreme ramps (1): {class_counts.get(1, 0):,} hours ({class_counts.get(1, 0)/len(df_features)*100:.1f}%)")

## 3. Train-Test Split (Time Series Aware)

In [ ]:
# Select feature columns (exclude target, datetime, and original columns)
feature_cols = [
    # Temporal
    'hour', 'day_of_week', 'month', 'is_weekend', 'hour_sin', 'hour_cos', 'season',
    # Lagged values
    'load_lag1', 'load_lag2', 'load_lag3',
    'wind_lag1', 'wind_lag2', 'wind_lag3',
    'solar_lag1', 'solar_lag2', 'solar_lag3',
    'netload_lag1', 'netload_lag2', 'netload_lag3',
    'ramp_lag1', 'ramp_lag2', 'ramp_lag3',
    # Trends
    'wind_change_3h', 'solar_change_3h', 'load_change_3h',
    'wind_change_6h', 'solar_change_6h',
    # Volatility
    'wind_volatility_6h', 'solar_volatility_6h', 'ramp_volatility_6h',
    # Ratios
    'renewable_penetration', 'wind_fraction', 'netload_normalized',
    # Flags
    'is_sunset', 'is_sunrise', 'is_summer',
    # Recent ramps
    'abs_ramp_lag1', 'abs_ramp_lag2',
]

# Remove rows with NaN (due to lagged features)
df_ml = df_features[feature_cols + ['target_extreme_ramp_next', 'target_ramp_magnitude', 'datetime']].dropna()

print(f"ML dataset shape after removing NaN: {df_ml.shape}")
print(f"Date range: {df_ml['datetime'].min()} to {df_ml['datetime'].max()}")

# Time series split: use first 80% for training, last 20% for testing
# This respects temporal order (no data leakage)
split_idx = int(len(df_ml) * 0.8)

train_df = df_ml.iloc[:split_idx]
test_df = df_ml.iloc[split_idx:]

X_train = train_df[feature_cols]
y_train_class = train_df['target_extreme_ramp_next']
y_train_reg = train_df['target_ramp_magnitude']

X_test = test_df[feature_cols]
y_test_class = test_df['target_extreme_ramp_next']
y_test_reg = test_df['target_ramp_magnitude']

print(f"\nTrain set: {len(X_train):,} samples ({train_df['datetime'].min()} to {train_df['datetime'].max()})")
print(f"Test set:  {len(X_test):,} samples ({test_df['datetime'].min()} to {test_df['datetime'].max()})")
print(f"\nTrain class balance: {y_train_class.value_counts().to_dict()}")
print(f"Test class balance:  {y_test_class.value_counts().to_dict()}")

## 4. Classification Model: Random Forest

In [ ]:
# Train Random Forest classifier
print("Training Random Forest Classifier...")
rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced',  # Handle class imbalance
    random_state=42,
    n_jobs=-1
)

rf_clf.fit(X_train, y_train_class)

# Predictions
y_pred_class = rf_clf.predict(X_test)
y_pred_proba = rf_clf.predict_proba(X_test)[:, 1]  # Probability of extreme ramp

# Evaluation metrics
print("\n" + "="*80)
print("RANDOM FOREST CLASSIFICATION RESULTS")
print("="*80)
print("\nClassification Report:")
print(classification_report(y_test_class, y_pred_class, 
                          target_names=['Normal', 'Extreme Ramp']))

# ROC-AUC
roc_auc = roc_auc_score(y_test_class, y_pred_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

# Confusion matrix
cm = confusion_matrix(y_test_class, y_pred_class)
print(f"\nConfusion Matrix:")
print(f"  True Negatives:  {cm[0,0]:,}")
print(f"  False Positives: {cm[0,1]:,}")
print(f"  False Negatives: {cm[1,0]:,}")
print(f"  True Positives:  {cm[1,1]:,}")

In [ ]:
# Plot ROC curve and confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC Curve
ax1 = axes[0]
fpr, tpr, thresholds = roc_curve(y_test_class, y_pred_proba)
ax1.plot(fpr, tpr, linewidth=2.5, label=f'Random Forest (AUC={roc_auc:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Guess')
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('ROC Curve: Extreme Ramp Prediction', fontsize=13, fontweight='bold')
ax1.grid(alpha=0.3)
ax1.legend(fontsize=11)

# Confusion Matrix
ax2 = axes[1]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2, 
           xticklabels=['Normal', 'Extreme'], yticklabels=['Normal', 'Extreme'])
ax2.set_xlabel('Predicted', fontsize=12)
ax2.set_ylabel('Actual', fontsize=12)
ax2.set_title('Confusion Matrix', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ml_classification_performance.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nSaved: {FIGURES_DIR / 'ml_classification_performance.png'}")

## 5. Feature Importance Analysis

In [ ]:
# Extract feature importances
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_clf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features:")
print(feature_importance.head(20).to_string(index=False))

# Plot feature importance
fig, ax = plt.subplots(figsize=(12, 10))

top_features = feature_importance.head(20)
ax.barh(range(len(top_features)), top_features['importance'], color='steelblue', alpha=0.8)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'])
ax.invert_yaxis()
ax.set_xlabel('Importance', fontsize=12)
ax.set_title('Top 20 Feature Importances: Extreme Ramp Prediction', 
            fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ml_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nSaved: {FIGURES_DIR / 'ml_feature_importance.png'}")

## 6. Regression Model: Predict Exact Ramp Magnitude

In [ ]:
# Train Random Forest regressor
print("Training Random Forest Regressor...")
rf_reg = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_reg.fit(X_train, y_train_reg)

# Predictions
y_pred_reg = rf_reg.predict(X_test)

# Evaluation metrics
mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2 = r2_score(y_test_reg, y_pred_reg)

print("\n" + "="*80)
print("RANDOM FOREST REGRESSION RESULTS")
print("="*80)
print(f"\nMean Absolute Error (MAE):  {mae:,.0f} MW/hr")
print(f"Root Mean Squared Error (RMSE): {rmse:,.0f} MW/hr")
print(f"R² Score: {r2:.4f}")

# Baseline comparison (predict mean)
baseline_mae = mean_absolute_error(y_test_reg, [y_train_reg.mean()] * len(y_test_reg))
improvement = (baseline_mae - mae) / baseline_mae * 100
print(f"\nBaseline MAE (predict mean): {baseline_mae:,.0f} MW/hr")
print(f"Improvement over baseline: {improvement:.1f}%")

In [ ]:
# Plot regression performance
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Predicted vs Actual
ax1 = axes[0, 0]
sample_idx = np.random.choice(len(y_test_reg), min(2000, len(y_test_reg)), replace=False)
ax1.scatter(y_test_reg.iloc[sample_idx], y_pred_reg[sample_idx], 
           alpha=0.3, s=10, color='steelblue')
max_val = max(y_test_reg.max(), y_pred_reg.max())
ax1.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect prediction')
ax1.set_xlabel('Actual Ramp Magnitude (MW/hr)', fontsize=11)
ax1.set_ylabel('Predicted Ramp Magnitude (MW/hr)', fontsize=11)
ax1.set_title(f'Predicted vs Actual (R²={r2:.3f})', fontsize=12, fontweight='bold')
ax1.grid(alpha=0.3)
ax1.legend()

# Residuals
ax2 = axes[0, 1]
residuals = y_test_reg.values - y_pred_reg
ax2.scatter(y_pred_reg, residuals, alpha=0.3, s=10, color='purple')
ax2.axhline(y=0, color='r', linestyle='--', linewidth=2)
ax2.set_xlabel('Predicted Ramp Magnitude (MW/hr)', fontsize=11)
ax2.set_ylabel('Residuals (MW/hr)', fontsize=11)
ax2.set_title('Residual Plot', fontsize=12, fontweight='bold')
ax2.grid(alpha=0.3)

# Error distribution
ax3 = axes[1, 0]
ax3.hist(residuals, bins=100, color='steelblue', alpha=0.7, edgecolor='black')
ax3.axvline(x=0, color='r', linestyle='--', linewidth=2)
ax3.set_xlabel('Prediction Error (MW/hr)', fontsize=11)
ax3.set_ylabel('Frequency', fontsize=11)
ax3.set_title(f'Error Distribution (MAE={mae:,.0f} MW/hr)', fontsize=12, fontweight='bold')
ax3.grid(alpha=0.3)

# Prediction accuracy by ramp magnitude bin
ax4 = axes[1, 1]
bins = np.arange(0, y_test_reg.max(), 1000)
bin_centers = (bins[:-1] + bins[1:]) / 2
bin_mae = []

for i in range(len(bins)-1):
    mask = (y_test_reg >= bins[i]) & (y_test_reg < bins[i+1])
    if mask.sum() > 0:
        bin_mae.append(mean_absolute_error(y_test_reg[mask], y_pred_reg[mask]))
    else:
        bin_mae.append(np.nan)

ax4.plot(bin_centers, bin_mae, 'o-', linewidth=2.5, markersize=6, color='darkred')
ax4.set_xlabel('Actual Ramp Magnitude (MW/hr)', fontsize=11)
ax4.set_ylabel('MAE in Bin (MW/hr)', fontsize=11)
ax4.set_title('Prediction Error by Ramp Magnitude', fontsize=12, fontweight='bold')
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ml_regression_performance.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nSaved: {FIGURES_DIR / 'ml_regression_performance.png'}")

## 7. Model Performance by Time Period

In [ ]:
# Analyze performance during critical periods
test_results = test_df.copy()
test_results['predicted_class'] = y_pred_class
test_results['predicted_proba'] = y_pred_proba
test_results['predicted_magnitude'] = y_pred_reg

# Performance by hour of day
hourly_performance = []
for hour in range(24):
    hour_mask = test_results['datetime'].dt.hour == hour
    if hour_mask.sum() > 0:
        y_true = test_results.loc[hour_mask, 'target_extreme_ramp_next']
        y_pred = test_results.loc[hour_mask, 'predicted_proba']
        
        if y_true.sum() > 0:  # Only if there are positive examples
            auc = roc_auc_score(y_true, y_pred)
        else:
            auc = np.nan
        
        hourly_performance.append({
            'hour': hour,
            'n_samples': hour_mask.sum(),
            'n_extreme': y_true.sum(),
            'roc_auc': auc,
        })

hourly_perf_df = pd.DataFrame(hourly_performance)

print("\nPERFORMANCE BY HOUR OF DAY:")
print(hourly_perf_df.to_string(index=False))

# Performance by season
seasonal_performance = []
season_names = {0: 'Winter', 1: 'Spring', 2: 'Summer', 3: 'Fall'}

for season_num, season_name in season_names.items():
    season_mask = test_results['datetime'].dt.month.isin(
        [12, 1, 2] if season_num == 0 else
        [3, 4, 5] if season_num == 1 else
        [6, 7, 8] if season_num == 2 else
        [9, 10, 11]
    )
    
    if season_mask.sum() > 0:
        y_true = test_results.loc[season_mask, 'target_extreme_ramp_next']
        y_pred = test_results.loc[season_mask, 'predicted_proba']
        
        if y_true.sum() > 0:
            auc = roc_auc_score(y_true, y_pred)
        else:
            auc = np.nan
        
        seasonal_performance.append({
            'season': season_name,
            'n_samples': season_mask.sum(),
            'n_extreme': y_true.sum(),
            'roc_auc': auc,
        })

seasonal_perf_df = pd.DataFrame(seasonal_performance)

print("\nPERFORMANCE BY SEASON:")
print(seasonal_perf_df.to_string(index=False))

In [ ]:
# Plot performance by time period
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Performance by hour
ax1 = axes[0]
ax1.plot(hourly_perf_df['hour'], hourly_perf_df['roc_auc'], 
        'o-', linewidth=2.5, markersize=8, color='steelblue')
ax1.axhline(y=0.5, color='red', linestyle='--', linewidth=1.5, label='Random guess')
ax1.axvspan(17, 20, alpha=0.2, color='orange', label='Sunset window')
ax1.set_xlabel('Hour of Day', fontsize=12)
ax1.set_ylabel('ROC-AUC', fontsize=12)
ax1.set_title('Model Performance by Hour', fontsize=13, fontweight='bold')
ax1.grid(alpha=0.3)
ax1.legend(fontsize=10)
ax1.set_xticks(range(0, 24, 3))
ax1.set_ylim(0.4, 1.0)

# Performance by season
ax2 = axes[1]
ax2.bar(seasonal_perf_df['season'], seasonal_perf_df['roc_auc'], 
       color=['skyblue', 'lightgreen', 'coral', 'gold'], alpha=0.8, edgecolor='black')
ax2.axhline(y=0.5, color='red', linestyle='--', linewidth=1.5, label='Random guess')
ax2.set_ylabel('ROC-AUC', fontsize=12)
ax2.set_title('Model Performance by Season', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.legend(fontsize=10)
ax2.set_ylim(0.4, 1.0)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ml_performance_by_time_period.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nSaved: {FIGURES_DIR / 'ml_performance_by_time_period.png'}")

## 8. Operational Application: Early Warning System

In [ ]:
# Simulate early warning system for last week of test data
last_week = test_results.tail(168)  # Last 7 days

# Define alert threshold (probability > 0.7 = high risk)
alert_threshold = 0.7
last_week['alert'] = last_week['predicted_proba'] > alert_threshold

# Calculate alert statistics
n_alerts = last_week['alert'].sum()
n_true_positives = ((last_week['alert']) & (last_week['target_extreme_ramp_next'] == 1)).sum()
n_false_positives = ((last_week['alert']) & (last_week['target_extreme_ramp_next'] == 0)).sum()
n_false_negatives = ((~last_week['alert']) & (last_week['target_extreme_ramp_next'] == 1)).sum()

precision = n_true_positives / n_alerts if n_alerts > 0 else 0
recall = n_true_positives / (n_true_positives + n_false_negatives) if (n_true_positives + n_false_negatives) > 0 else 0

print("\n" + "="*80)
print("EARLY WARNING SYSTEM SIMULATION (Last Week of Test Data)")
print("="*80)
print(f"\nAlert threshold: {alert_threshold:.1%} probability")
print(f"\nTotal alerts issued: {n_alerts} ({n_alerts/len(last_week)*100:.1f}% of hours)")
print(f"  True positives (caught extreme events): {n_true_positives}")
print(f"  False positives (false alarms): {n_false_positives}")
print(f"  False negatives (missed extreme events): {n_false_negatives}")
print(f"\nPrecision: {precision:.1%} (of alerts, {precision:.1%} were real)")
print(f"Recall: {recall:.1%} (caught {recall:.1%} of extreme events)")

# Cost-benefit analysis
cost_false_alarm = 5000  # $ per false alarm (unnecessary reserve activation)
cost_missed_event = 50000  # $ per missed extreme ramp (emergency measures)
benefit_caught_event = 40000  # $ saved per caught event (planned response)

total_cost = (n_false_positives * cost_false_alarm + 
             n_false_negatives * cost_missed_event)
total_benefit = n_true_positives * benefit_caught_event
net_value = total_benefit - total_cost

print(f"\nCOST-BENEFIT (Hypothetical):")
print(f"  Total cost (false alarms + missed events): ${total_cost:,}")
print(f"  Total benefit (caught events): ${total_benefit:,}")
print(f"  Net value: ${net_value:,}")

In [ ]:
# Plot early warning timeline
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# Actual ramp magnitude
ax1 = axes[0]
ax1.plot(last_week['datetime'], last_week['target_ramp_magnitude'], 
        linewidth=1.5, color='black', label='Actual ramp')
ax1.axhline(y=ramp_threshold, color='red', linestyle='--', linewidth=2, 
           label=f'Extreme threshold ({ramp_threshold:,.0f} MW/hr)')
ax1.set_ylabel('Ramp (MW/hr)', fontsize=11)
ax1.set_title('Early Warning System: Last Week of Test Data', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# Predicted probability
ax2 = axes[1]
ax2.plot(last_week['datetime'], last_week['predicted_proba'], 
        linewidth=1.5, color='steelblue', label='Predicted probability')
ax2.axhline(y=alert_threshold, color='red', linestyle='--', linewidth=2, 
           label=f'Alert threshold ({alert_threshold:.1%})')
ax2.fill_between(last_week['datetime'], 0, 1, 
                 where=last_week['alert'], alpha=0.3, color='red', label='Alert issued')
ax2.set_ylabel('Probability', fontsize=11)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)
ax2.set_ylim(0, 1)

# Alert outcomes
ax3 = axes[2]
true_positives = last_week['alert'] & (last_week['target_extreme_ramp_next'] == 1)
false_positives = last_week['alert'] & (last_week['target_extreme_ramp_next'] == 0)
false_negatives = (~last_week['alert']) & (last_week['target_extreme_ramp_next'] == 1)

ax3.scatter(last_week.loc[true_positives, 'datetime'], 
           [1]*true_positives.sum(), 
           color='green', s=100, marker='o', label='True Positive (caught)', zorder=5)
ax3.scatter(last_week.loc[false_positives, 'datetime'], 
           [0.5]*false_positives.sum(), 
           color='orange', s=100, marker='x', label='False Positive (false alarm)', zorder=5)
ax3.scatter(last_week.loc[false_negatives, 'datetime'], 
           [0]*false_negatives.sum(), 
           color='red', s=100, marker='s', label='False Negative (missed)', zorder=5)

ax3.set_xlabel('Date', fontsize=11)
ax3.set_ylabel('Alert Outcome', fontsize=11)
ax3.legend(fontsize=10, loc='upper right')
ax3.grid(alpha=0.3)
ax3.set_ylim(-0.2, 1.2)
ax3.set_yticks([0, 0.5, 1])
ax3.set_yticklabels(['Miss', 'False Alarm', 'Catch'])

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ml_early_warning_timeline.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nSaved: {FIGURES_DIR / 'ml_early_warning_timeline.png'}")

## 9. Summary and Conclusions

In [ ]:
# Generate summary report
summary_data = {
    'Metric': [
        'Classification ROC-AUC',
        'Classification Precision (Extreme class)',
        'Classification Recall (Extreme class)',
        'Regression MAE (MW/hr)',
        'Regression RMSE (MW/hr)',
        'Regression R²',
        'Top Feature',
        'Early Warning Precision',
        'Early Warning Recall',
    ],
    'Value': [
        f'{roc_auc:.4f}',
        classification_report(y_test_class, y_pred_class, output_dict=True)['1']['precision'],
        classification_report(y_test_class, y_pred_class, output_dict=True)['1']['recall'],
        f'{mae:,.0f}',
        f'{rmse:,.0f}',
        f'{r2:.4f}',
        feature_importance.iloc[0]['feature'],
        f'{precision:.2%}',
        f'{recall:.2%}',
    ]
}

summary_df = pd.DataFrame(summary_data)

print("\n" + "="*80)
print("MACHINE LEARNING RAMPING RISK PREDICTION: SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))

# Save report
summary_df.to_csv(PROCESSED_DIR / 'ml_prediction_summary.csv', index=False)
print(f"\n\nSummary saved to: {PROCESSED_DIR / 'ml_prediction_summary.csv'}")

print("\n" + "="*80)
print("KEY FINDINGS:")
print("="*80)
print("\n1. CLASSIFICATION PERFORMANCE:")
print(f"   - ROC-AUC of {roc_auc:.3f} indicates {'excellent' if roc_auc > 0.9 else 'good' if roc_auc > 0.8 else 'moderate'} predictive power")
print("   - Can predict extreme ramping events 1 hour ahead")

print("\n2. MOST IMPORTANT FEATURES:")
for i, row in feature_importance.head(5).iterrows():
    print(f"   {i+1}. {row['feature']} ({row['importance']:.4f})")

print("\n3. OPERATIONAL VALUE:")
print(f"   - Early warning system with {alert_threshold:.0%} threshold achieves:")
print(f"     • {precision:.1%} precision (low false alarm rate)")
print(f"     • {recall:.1%} recall (catches most extreme events)")
print(f"   - Estimated net value: ${net_value:,} per week")

print("\n4. REGRESSION PERFORMANCE:")
print(f"   - MAE of {mae:,.0f} MW/hr represents {mae/ramp_threshold*100:.1f}% of extreme threshold")
print(f"   - {improvement:.1f}% improvement over naive baseline")

print("\n" + "="*80)
print("RECOMMENDATIONS:")
print("="*80)
print("\n1. Deploy as real-time early warning system for grid operators")
print("2. Integrate with reserve scheduling algorithms")
print("3. Consider ensemble models (combining RF + Gradient Boosting)")
print("4. Extend prediction horizon to 2-3 hours ahead")
print("5. Incorporate real-time weather forecast data for improved accuracy")